In [1]:
import pandas as pd
import sys
import os
import numpy as np
import scanpy as sc

In [2]:
adata = sc.read_h5ad(r"D:\Trapecar\250307_gut_liver_blood_ultimate_annotated.h5ad")

In [3]:
adata_TCR = adata[adata.obs['chain_pairing'].isin(["single pair", "extra VJ","extra VDJ","two full chains"]),:]
adata_ab = adata_TCR[adata_TCR.obs['general type'].isin(['TCRab CD4','TCRab CD8aa','TCRab CD8ab']),:]
adata_ab.obs['clone_code'] = adata_ab.obs['TRAV'].astype(str).map(str)+' '+adata_ab.obs['TRBV'].astype(str).map(str)+' '+adata_ab.obs['cdr3a'].astype(str).map(str)+' '+ adata_ab.obs['cdr3b'].astype(str).map(str)
adata_ab.obs['subject:condition']= adata_ab.obs['Donor ID'].astype(str).map(str) + ':' + adata_ab.obs['tissue+celltype'].astype(str).map(str)

C:\Users\andre\AppData\Local\Temp\ipykernel_29516\3865754011.py:3: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_ab.obs['clone_code'] = adata_ab.obs['TRAV'].astype(str).map(str)+' '+adata_ab.obs['TRBV'].astype(str).map(str)+' '+adata_ab.obs['cdr3a'].astype(str).map(str)+' '+ adata_ab.obs['cdr3b'].astype(str).map(str)


In [4]:
# from conga/tcrdist
clone_counts= pd.read_csv(r"G:\My Drive\result\publication\cellreport\revision\conga\gut_liver_TRM_clones_with0.tsv",sep = '\t',index_col = 0)
temp_dict_df = clone_counts[clone_counts['clone_size']>0][['clone_id','va_gene','vb_gene','cdr3a','cdr3b']]
temp_dict_df['clone_code'] = temp_dict_df['va_gene'].astype(str).map(str) + ' ' + temp_dict_df['vb_gene'].astype(str).map(str) + ' ' + temp_dict_df['cdr3a'].astype(str).map(str) + ' ' + temp_dict_df['cdr3b'].astype(str).map(str)
clone_replace_dict = temp_dict_df[['clone_id','clone_code']].set_index('clone_code').sort_index()['clone_id'].to_dict()

In [29]:
os.chdir(r"G:\My Drive\result\publication\cellreport\revision\GLIPH2")
names = ['TCRab CD4','TCRab CD8aa','TCRab CD8ab']
for i in names:
    adata_slice = adata_ab[adata_ab.obs['general type'] == i,:]

    clone_df = adata_slice.obs[['subject:condition','TRAV', 'TRBV','TRBJ', 'cdr3a', 'cdr3b','clone_code']]
    clone_counts = clone_df.groupby(['subject:condition','clone_code']).size().reset_index(name='clone frequency')
    clone_counts[['TRAV','TRBV','cdr3a','cdr3b']] = clone_counts['clone_code'].str.split(' ', expand=True)
    
    Jmap = clone_df[['TRBJ','clone_code']].drop_duplicates()
    Jmap = Jmap.set_index('clone_code')
    Jmap_bdict = Jmap['TRBJ'].to_dict()
    clone_counts['TRBJ'] =clone_counts['clone_code'].replace(Jmap_bdict)

    clone_counts = clone_counts[clone_counts['clone frequency'] >= 1]
    clone_counts['clone_id'] = clone_counts['clone_code'].map(clone_replace_dict)
    clone_counts[['cdr3b','TRBV','TRBJ','cdr3a','subject:condition','clone frequency']].to_csv(i+'_GLIPH2.tsv', sep="\t",index = False, header = False)

In [28]:
clone_counts

,subject:condition,clone_code,clone frequency,TRAV,TRBV,cdr3a,cdr3b,clone_id
0,Donor AJD3280:IEL TCRab CD8ab TRM,TRAV1-1 TRBV11-1 CAVNTNAGKSTF CASSLPNEKLFF,1,TRAV1-1,TRBV11-1,CAVNTNAGKSTF,CASSLPNEKLFF,clonotype6
1,Donor AJD3280:IEL TCRab CD8ab TRM,TRAV1-1 TRBV11-2 CAAHTNAGKSTF CASTSRDRGLHEQYF,1,TRAV1-1,TRBV11-2,CAAHTNAGKSTF,CASTSRDRGLHEQYF,clonotype7
2,Donor AJD3280:IEL TCRab CD8ab TRM,TRAV1-1 TRBV11-2 CAVETNAGKSTF CASSDRGQGGANVLTF,2,TRAV1-1,TRBV11-2,CAVETNAGKSTF,CASSDRGQGGANVLTF,clonotype9
3,Donor AJD3280:IEL TCRab CD8ab TRM,TRAV1-1 TRBV11-2 CAVSTNAGKSTF CASSVPNEKLFF,3,TRAV1-1,TRBV11-2,CAVSTNAGKSTF,CASSVPNEKLFF,clonotype11
4,Donor AJD3280:IEL TCRab CD8ab TRM,TRAV1-1 TRBV11-2 CAVYSGYSTLTF CASSLVIAGTSDTQYF,1,TRAV1-1,TRBV11-2,CAVYSGYSTLTF,CASSLVIAGTSDTQYF,clonotype12
...,...,...,...,...,...,...,...,...
6707,Donor AJKQ118:PB TCRab CD8ab Teff,TRAV8-6 TRBV28 CAVATGANNLFF CAAARGYGNQPQHF,1,TRAV8-6,TRBV28,CAVATGANNLFF,CAAARGYGNQPQHF,clonotype17694
6708,Donor AJKQ118:PB TCRab CD8ab Teff,TRAV8-6 TRBV5-1 CAVALKTSYDKVIF CASSSRVNRETQYF,1,TRAV8-6,TRBV5-1,CAVALKTSYDKVIF,CASSSRVNRETQYF,clonotype17753
6709,Donor AJKQ118:PB TCRab CD8ab Teff,TRAV8-6 TRBV5-1 CAVRISGGSYIPTF CASSFLYNSPLHF,1,TRAV8-6,TRBV5-1,CAVRISGGSYIPTF,CASSFLYNSPLHF,clonotype17763
6710,Donor AJKQ118:PB TCRab CD8ab Teff,TRAV8-6 TRBV6-6 CAVSDLYNAGNMLTF CASKGTSDTEAFF,1,TRAV8-6,TRBV6-6,CAVSDLYNAGNMLTF,CASKGTSDTEAFF,clonotype17843


In [26]:
adata_ab.obs['general type']

AAACCTGAGCGCCTCA-1-3-PB       TCRab CD4
AAACCTGAGGCAAAGA-1-3-PB     TCRab CD8ab
AAACCTGAGTTATCGC-1-3-PB       TCRab CD4
AAACCTGCAAGCGATG-1-3-PB       TCRab CD4
AAACCTGGTTACGTCA-1-3-PB       TCRab CD4
                               ...     
TTTGTCAAGAACTGTA-1-5-IEL    TCRab CD8ab
TTTGTCAAGGGAGTAA-1-5-IEL    TCRab CD8ab
TTTGTCAAGTTGTCGT-1-5-IEL      TCRab CD4
TTTGTCACAATGACCT-1-5-IEL    TCRab CD8ab
TTTGTCACATCGATTG-1-5-IEL    TCRab CD8ab
Name: general type, Length: 26297, dtype: category
Categories (3, object): ['TCRab CD4', 'TCRab CD8aa', 'TCRab CD8ab']

### Mapping GLIPH2 reulst back

#### CD4

In [2]:
CD4_GLIPH2 = pd.read_csv(r"C:\Users\andre\Documents\GitHub\gut-liver-TRM\Revision\GLIPH2\GLIPH2_results\CD4_GLIPH2.csv",index_col = 0).iloc[:,0:18]

In [ ]:
CD4_GLIPH2['celltype'] = CD4_GLIPH2['Sample'].str.split(':').str[1]
pattern_counts = CD4_GLIPH2.pivot_table(index = 'type', columns='celltype', values = 'Freq', aggfunc="sum", fill_value=0)
# motif_id = [i.startswith('motif') for i in pattern_counts.index]
# pattern_counts = pattern_counts[motif_id]

In [ ]:
# pattern_counts

celltype,IEL TCRab CD4 FOXP3+ Treg,IEL TCRab CD4 Mobile TRM,IEL TCRab CD4 TRM,L TCRab CD4 FOXP3+ Treg,L TCRab CD4 Naive/TCM,L TCRab CD4 TCM,L TCRab CD4 TRM,LP TCRab CD4 FOXP3+ Treg,LP TCRab CD4 Mobile TRM,LP TCRab CD4 Naive/TCM,LP TCRab CD4 Poised TCM,LP TCRab CD4 TRM,LP TCRab CD4 Tph,PB TCRab CD4 FOXP3+ Treg,PB TCRab CD4 Naive/TCM,PB TCRab CD4 TCM
type,,,,,,,,,,,,,,,,
motif-DHNQ,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
motif-EYGN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
motif-HWNT motif-HWN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,4.0,0.0,0.0,4.0,0.0
motif-IILA,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
motif-KIK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
motif-LKVS,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
motif-LPNA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0
motif-LYLR,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
motif-QEQP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,2.0,0.0


In [47]:
nodes = pattern_counts.columns
migration_counts = pd.DataFrame(index=list(nodes), columns=list(nodes))
for nodes1 in nodes:
    for nodes2 in nodes:
        migration_counts.loc[nodes1, nodes2] = np.sum((pattern_counts[nodes1] > 0) & (pattern_counts[nodes2] > 0))
        if nodes1 == nodes2:
            migration_counts.loc[nodes1, nodes2] = 0
migration_counts

,IEL TCRab CD4 FOXP3+ Treg,IEL TCRab CD4 Mobile TRM,IEL TCRab CD4 TRM,L TCRab CD4 FOXP3+ Treg,L TCRab CD4 Naive/TCM,L TCRab CD4 TCM,L TCRab CD4 TRM,LP TCRab CD4 FOXP3+ Treg,LP TCRab CD4 Mobile TRM,LP TCRab CD4 Naive/TCM,LP TCRab CD4 Poised TCM,LP TCRab CD4 TRM,LP TCRab CD4 Tph,PB TCRab CD4 FOXP3+ Treg,PB TCRab CD4 Naive/TCM,PB TCRab CD4 TCM
IEL TCRab CD4 FOXP3+ Treg,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0
IEL TCRab CD4 Mobile TRM,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
IEL TCRab CD4 TRM,0,0,0,0,1,0,1,1,0,0,0,4,2,2,3,0
L TCRab CD4 FOXP3+ Treg,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
L TCRab CD4 Naive/TCM,0,0,1,0,0,0,0,0,0,0,0,1,0,1,2,0
L TCRab CD4 TCM,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
L TCRab CD4 TRM,0,0,1,0,0,0,0,2,0,0,0,3,0,2,2,0
LP TCRab CD4 FOXP3+ Treg,0,0,1,0,0,0,2,0,1,0,1,2,0,2,1,0
LP TCRab CD4 Mobile TRM,0,0,0,0,0,0,0,1,0,0,1,1,1,0,2,0
LP TCRab CD4 Naive/TCM,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0


In [41]:
migration_counts[migration_counts<np.percentile(migration_counts.values.flatten(),50)] = 0

In [ ]:
migration_counts.to_csv("G:/My Drive/result/publication/cellreport/revision/GLIPH2/GLIPH2_results/CD4_GLIPH2_migration_counts.csv")

#### now do the same for CD8ab

In [ ]:
CD8_GLIPH2 = pd.read_csv(r"C:\Users\andre\Documents\GitHub\gut-liver-TRM\Revision\GLIPH2\GLIPH2_results\CD8_GLIPH2.csv",index_col = 0).iloc[:,0:18]
CD8_GLIPH2['celltype'] = CD8_GLIPH2['Sample'].str.split(':').str[1]
pattern_counts = CD8_GLIPH2.pivot_table(index = 'type', columns='celltype', values = 'Freq', aggfunc="sum", fill_value=0)
# motif_id = [i.startswith('motif') for i in pattern_counts.index]
# pattern_counts = pattern_counts[motif_id]
# pattern_counts
nodes = pattern_counts.columns
migration_counts = pd.DataFrame(index=list(nodes), columns=list(nodes))
for nodes1 in nodes:
    for nodes2 in nodes:
        migration_counts.loc[nodes1, nodes2] = np.sum((pattern_counts[nodes1] > 0) & (pattern_counts[nodes2] > 0))
        if nodes1 == nodes2:
            migration_counts.loc[nodes1, nodes2] = 0
migration_counts
migration_counts[migration_counts<np.percentile(migration_counts.values.flatten(),50)] = 0
migration_counts.to_csv("G:/My Drive/result/publication/cellreport/revision/GLIPH2/GLIPH2_results/CD8_GLIPH2_migration_counts.csv")